# Full-History Dataset Verification

Verifies the merged session-aligned panel:

- duplicate `ticker + date` rows
- equal number of dates per ticker
- identical date set for every ticker
- date ordering and date gaps
- schema compatibility with the original nine-ticker dataset
- missing-value rates by ticker and source column group

The panel is session-aligned, so weekends and market holidays are expected to be absent. The main completeness check is whether every ticker has the same trading-session date grid.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASETS_DIR = PROJECT_ROOT / "data" / "datasets"

DATASET_PATH = DATASETS_DIR / "stock_panel_nine_tickers_session_aligned_full_history_raw.csv"
REFERENCE_DATASET_PATH = DATASETS_DIR / "stock_panel_nine_tickers_session_aligned_raw.csv"

OUTPUT_PREFIX = DATASET_PATH.with_suffix("").name.replace("_raw", "")
VERIFICATION_REPORT_PATH = DATASETS_DIR / f"{OUTPUT_PREFIX}_verification_report.csv"
DATE_GAPS_PATH = DATASETS_DIR / f"{OUTPUT_PREFIX}_date_gap_audit.csv"
MISSING_GRID_PATH = DATASETS_DIR / f"{OUTPUT_PREFIX}_missing_ticker_date_grid.csv"
MISSING_RATES_PATH = DATASETS_DIR / f"{OUTPUT_PREFIX}_missing_rates_by_ticker.csv"
TICKER_SUMMARY_PATH = DATASETS_DIR / f"{OUTPUT_PREFIX}_ticker_date_summary.csv"

EXPECTED_TICKERS = ["AAPL", "AMD", "AMZN", "GOOGL", "META", "MSFT", "NFLX", "NVDA", "TSLA"]
EXPECTED_START_DATE = pd.Timestamp("2021-01-04")
EXPECTED_END_DATE = pd.Timestamp("2025-12-31")

FAIL_FAST = False

DATASET_PATH

In [ ]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATASET_PATH}")

df = pd.read_csv(DATASET_PATH, parse_dates=["date"])
df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()

print(f"Loaded {DATASET_PATH}")
print(f"shape={df.shape}")
print(f"date_min={df['date'].min().date()} date_max={df['date'].max().date()}")
print(f"tickers={sorted(df['ticker'].dropna().unique().tolist())}")

df.head()

In [ ]:
checks: list[dict[str, object]] = []


def add_check(name: str, status: str, detail: str, value: object = None) -> None:
    checks.append(
        {
            "check": name,
            "status": status,
            "value": value,
            "detail": detail,
        }
    )


def add_boolean_check(name: str, condition: bool, pass_detail: str, fail_detail: str, value: object = None) -> None:
    add_check(name, "PASS" if condition else "FAIL", pass_detail if condition else fail_detail, value)


def make_empty_grid_frame() -> pd.DataFrame:
    return pd.DataFrame(columns=["ticker", "date"])


required_columns = ["date", "ticker", "company_name", "stock_price", "stock_volume"]
missing_required_columns = [column for column in required_columns if column not in df.columns]
add_boolean_check(
    "required_columns_present",
    not missing_required_columns,
    "All required identity and stock columns are present.",
    f"Missing required columns: {missing_required_columns}",
    ", ".join(missing_required_columns),
)

add_boolean_check(
    "date_parse_success",
    df["date"].notna().all(),
    "All date values parsed successfully.",
    f"Unparsed date rows: {int(df['date'].isna().sum())}",
    int(df["date"].isna().sum()),
)

actual_tickers = sorted(df["ticker"].dropna().unique().tolist())
add_boolean_check(
    "expected_tickers",
    actual_tickers == EXPECTED_TICKERS,
    "Ticker universe matches EXPECTED_TICKERS.",
    f"Expected {EXPECTED_TICKERS}, got {actual_tickers}.",
    ", ".join(actual_tickers),
)

date_min = df["date"].min()
date_max = df["date"].max()
add_boolean_check(
    "expected_date_range",
    date_min == EXPECTED_START_DATE and date_max == EXPECTED_END_DATE,
    "Date range matches expected full-history range.",
    f"Expected {EXPECTED_START_DATE.date()} to {EXPECTED_END_DATE.date()}, got {date_min.date()} to {date_max.date()}.",
    f"{date_min.date()} -> {date_max.date()}",
)

duplicate_mask = df.duplicated(["ticker", "date"], keep=False)
duplicate_rows = df.loc[duplicate_mask, ["ticker", "date"]].sort_values(["ticker", "date"])
add_boolean_check(
    "no_duplicate_ticker_date_rows",
    duplicate_rows.empty,
    "No duplicate ticker-date rows found.",
    f"Duplicate ticker-date rows found: {len(duplicate_rows)}.",
    int(len(duplicate_rows)),
)

unique_dates = pd.Index(pd.to_datetime(sorted(df["date"].dropna().unique())), name="date")
unique_tickers = pd.Index(actual_tickers, name="ticker")
expected_grid_rows = len(unique_dates) * len(unique_tickers)
add_boolean_check(
    "row_count_matches_complete_grid",
    len(df) == expected_grid_rows,
    "Row count equals unique_dates * unique_tickers.",
    f"Expected {expected_grid_rows} rows from complete grid, got {len(df)}.",
    int(len(df)),
)

actual_grid = pd.MultiIndex.from_frame(df[["ticker", "date"]].drop_duplicates())
expected_grid = pd.MultiIndex.from_product([unique_tickers, unique_dates], names=["ticker", "date"])
missing_grid = expected_grid.difference(actual_grid)
missing_grid_df = missing_grid.to_frame(index=False) if len(missing_grid) else make_empty_grid_frame()
add_boolean_check(
    "complete_ticker_date_grid",
    missing_grid_df.empty,
    "Every ticker has every date in the shared date grid.",
    f"Missing ticker-date pairs: {len(missing_grid_df)}.",
    int(len(missing_grid_df)),
)

ticker_date_sets = {
    ticker: pd.Index(pd.to_datetime(sorted(group["date"].dropna().unique())))
    for ticker, group in df.groupby("ticker")
}
tickers_with_different_dates = []
for ticker, ticker_dates in ticker_date_sets.items():
    if not ticker_dates.equals(unique_dates):
        tickers_with_different_dates.append(ticker)

add_boolean_check(
    "same_date_set_for_each_ticker",
    not tickers_with_different_dates,
    "All tickers share the exact same date set.",
    f"Tickers with different date sets: {tickers_with_different_dates}",
    ", ".join(tickers_with_different_dates),
)

is_monotonic_by_ticker = df.sort_values(["ticker", "date"]).groupby("ticker")["date"].apply(lambda s: s.is_monotonic_increasing).all()
add_boolean_check(
    "dates_monotonic_within_ticker",
    bool(is_monotonic_by_ticker),
    "Dates are monotonic increasing within each ticker after sorting.",
    "At least one ticker has non-monotonic dates.",
    bool(is_monotonic_by_ticker),
)

weekend_dates = unique_dates[pd.Series(unique_dates).dt.weekday.to_numpy() >= 5]
add_boolean_check(
    "no_weekend_session_dates",
    len(weekend_dates) == 0,
    "No weekend dates found in the session grid.",
    f"Weekend dates found: {len(weekend_dates)}.",
    int(len(weekend_dates)),
)

stock_missing = df[["stock_price", "stock_volume"]].isna().sum().sum()
add_boolean_check(
    "stock_columns_not_missing",
    int(stock_missing) == 0,
    "stock_price and stock_volume have no missing values.",
    f"Missing stock values found: {int(stock_missing)}.",
    int(stock_missing),
)

if REFERENCE_DATASET_PATH.exists():
    reference_columns = pd.read_csv(REFERENCE_DATASET_PATH, nrows=1).columns.tolist()
    current_columns = df.columns.tolist()
    add_boolean_check(
        "schema_matches_reference_dataset",
        current_columns == reference_columns,
        "Column order matches the original nine-ticker session-aligned dataset.",
        "Column order differs from the original nine-ticker session-aligned dataset.",
        len(current_columns),
    )
else:
    add_check("schema_matches_reference_dataset", "WARN", f"Reference dataset not found: {REFERENCE_DATASET_PATH}")

verification_report = pd.DataFrame(checks)
verification_report

In [ ]:
ticker_summary = (
    df.groupby("ticker")
    .agg(
        rows=("date", "size"),
        unique_dates=("date", "nunique"),
        date_min=("date", "min"),
        date_max=("date", "max"),
        stock_price_missing=("stock_price", lambda s: int(s.isna().sum())),
        stock_volume_missing=("stock_volume", lambda s: int(s.isna().sum())),
    )
    .reset_index()
)
ticker_summary["date_min"] = ticker_summary["date_min"].dt.date.astype(str)
ticker_summary["date_max"] = ticker_summary["date_max"].dt.date.astype(str)
ticker_summary["matches_global_date_count"] = ticker_summary["unique_dates"].eq(len(unique_dates))

ticker_summary.to_csv(TICKER_SUMMARY_PATH, index=False)
ticker_summary

In [ ]:
date_gap_audit = pd.DataFrame({"date": unique_dates})
date_gap_audit["previous_date"] = date_gap_audit["date"].shift(1)
date_gap_audit["gap_days"] = (date_gap_audit["date"] - date_gap_audit["previous_date"]).dt.days
date_gap_audit = date_gap_audit[date_gap_audit["gap_days"].fillna(1).gt(1)].copy()
date_gap_audit["previous_date"] = date_gap_audit["previous_date"].dt.date.astype(str)
date_gap_audit["date"] = date_gap_audit["date"].dt.date.astype(str)

business_day_index = pd.bdate_range(unique_dates.min(), unique_dates.max())
missing_business_days = pd.Index(business_day_index).difference(unique_dates)
extra_non_business_days = unique_dates.difference(pd.Index(business_day_index))

add_check(
    "calendar_day_gaps",
    "INFO",
    "Calendar gaps are expected because the panel is trading-session aligned, not calendar-day aligned.",
    int(len(date_gap_audit)),
)
add_check(
    "missing_weekdays_vs_simple_business_calendar",
    "INFO",
    "Missing weekdays are usually US market holidays; this uses pandas business days, not an exchange holiday calendar.",
    int(len(missing_business_days)),
)
add_check(
    "extra_non_business_dates",
    "WARN" if len(extra_non_business_days) else "PASS",
    "Dates outside simple Monday-Friday business days.",
    int(len(extra_non_business_days)),
)

date_gap_audit.to_csv(DATE_GAPS_PATH, index=False)
date_gap_audit.head(20)

In [ ]:
value_columns = [column for column in df.columns if column not in ["date", "ticker", "company_name"]]
missing_rates_by_ticker = (
    df.groupby("ticker")[value_columns]
    .agg(lambda s: float(s.isna().mean()))
    .reset_index()
)

important_alt_columns = [
    "google_trends_score",
    "gdelt_articles",
    "gdelt_robust",
    "gdelt_sentiment_score",
    "subm_reddit_posts",
    "comm_reddit_posts",
]
important_alt_columns = [column for column in important_alt_columns if column in df.columns]
important_missing_max = missing_rates_by_ticker[important_alt_columns].max().max() if important_alt_columns else 0.0
add_check(
    "important_source_columns_mostly_present",
    "PASS" if important_missing_max <= 0.01 else "WARN",
    "Important aligned source columns have <=1% missing rate for every ticker."
    if important_missing_max <= 0.01
    else f"Max missing rate in important source columns is {important_missing_max:.4f}; inspect missing_rates_by_ticker.",
    float(important_missing_max),
)

missing_rates_by_ticker.to_csv(MISSING_RATES_PATH, index=False)
missing_rates_by_ticker[["ticker"] + important_alt_columns]

In [ ]:
reddit_coverage = df.groupby("ticker").agg(
    rows=("date", "size"),
    subm_days_with_posts=("subm_reddit_posts", lambda s: int((s.fillna(0) > 0).sum())),
    subm_post_coverage=("subm_reddit_posts", lambda s: float((s.fillna(0) > 0).mean())),
    comm_days_with_posts=("comm_reddit_posts", lambda s: int((s.fillna(0) > 0).sum())),
    comm_post_coverage=("comm_reddit_posts", lambda s: float((s.fillna(0) > 0).mean())),
).reset_index()

reddit_coverage

In [ ]:
missing_grid_df.to_csv(MISSING_GRID_PATH, index=False)
verification_report = pd.DataFrame(checks)
verification_report.to_csv(VERIFICATION_REPORT_PATH, index=False)

print(f"Saved verification report to {VERIFICATION_REPORT_PATH}")
print(f"Saved ticker summary to {TICKER_SUMMARY_PATH}")
print(f"Saved date gap audit to {DATE_GAPS_PATH}")
print(f"Saved missing ticker-date grid to {MISSING_GRID_PATH}")
print(f"Saved missing rates by ticker to {MISSING_RATES_PATH}")

status_counts = verification_report["status"].value_counts().to_dict()
print(f"status_counts={status_counts}")

failures = verification_report[verification_report["status"].eq("FAIL")]
if FAIL_FAST and not failures.empty:
    raise AssertionError(f"Dataset verification failed: {failures['check'].tolist()}")

verification_report